# TradeFlow AI — nb4_eval (Pillar 5: Final Evaluation)

**Tujuan**: Menguji model OLM LoRA yang sudah di-finetune (dari `nb3`) terhadap 8 Dokumen B/L Asli (Ground Truth).
Menghitung skor ANLS per field untuk memastikan NFR-007 (Akurasi >= 85%) tercapai.

In [ ]:
!pip install -q rapidfuzz transformers peft accelerate pillow

In [ ]:
import os, json, time
from pathlib import Path
import torch
from rapidfuzz import fuzz
import numpy as np
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def calculate_anls(gt_val: str, pred_val: str, threshold=0.5) -> float:
    if not gt_val and not pred_val: return 1.0
    if not gt_val or not pred_val: return 0.0
    
    gt_str = str(gt_val).lower().strip()
    pred_str = str(pred_val).lower().strip()
    
    ed = 1.0 - (fuzz.ratio(gt_str, pred_str) / 100.0)
    score = 1.0 - ed
    return score if score >= threshold else 0.0

# ── KAGGLE PATHS ──
# Pastikan Anda sudah Add Data: dataset 'tradeflow-real-docs'
REAL_DOCS_DIR = Path('/kaggle/input/tradeflow-real-docs')
GT_PATH = REAL_DOCS_DIR / 'TradeFlow_GroundTruth_v5.2.json'

BASE_MODEL_ID = 'allenai/olmOCR-7B-0225-preview'
# Ganti dengan path output LoRA dari nb3 (misal: /kaggle/input/notebooks/xxx/olmocr-tradeflow-lora/best)
# Atau nama repo HuggingFace Anda: 'nama-org/olm-ocr-cipl-v1'
LORA_ADAPTER_ID = 'GANTI_DENGAN_PATH_ATAU_REPO_LORA'


In [ ]:
print("Memuat Base Model & LoRA Adapter...")
print("Jika error, pastikan LORA_ADAPTER_ID di atas sudah diganti.")

# SIMULASI EVALUASI (Ganti 'True' menjadi 'False' untuk menjalankan inferensi nyata)
IS_SIMULATION = True

if not IS_SIMULATION:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,
        device_map='auto'
    )
    model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_ID)
    model.eval()
    print("Model siap!")
else:
    print("[MODE SIMULASI] Model belum dimuat.")


In [ ]:
def evaluate_all():
    if not GT_PATH.exists():
        print(f"File GT tidak ditemukan: {GT_PATH}")
        return
        
    with open(GT_PATH, 'r') as f:
        gt_data = json.load(f)
        
    field_scores = {k: [] for k in ['nomorBl', 'tglBl', 'pelabuhan_muat', 'pelabuhan_bongkar', 'container_no', 'beratKotor', 'hs_code']}
    
    print("\n=== MEMULAI EVALUASI ===\n")
    for pdf_name, gt_fields in gt_data.items():
        print(f"Evaluating: {pdf_name}")
        
        if IS_SIMULATION:
            # Mock prediksi dengan tingkat akurasi ~90%
            pred_json = gt_fields.copy()
            # Inject sedikit error untuk realistis
            if np.random.random() < 0.2: 
                pred_json['container_no'] = pred_json.get('container_no', '') + 'X'
        else:
            # INFERENSI NYATA KE MODEL OLM
            # prompt = ...
            # outputs = model.generate(...)
            # pred_json = json.loads(...)
            pred_json = {} # Ganti dengan kode inferensi asli
            
        for field in field_scores.keys():
            gt_val = str(gt_fields.get(field, ''))
            pred_val = str(pred_json.get(field, ''))
            score = calculate_anls(gt_val, pred_val)
            field_scores[field].append(score)
            print(f"  - {field}: ANLS {score:.3f} | GT: '{gt_val}' | Pred: '{pred_val}'")
            
    print("\n=== HASIL AKHIR (ANLS SCORE) ===")
    total_scores = []
    for field, scores in field_scores.items():
        avg = np.mean(scores)
        total_scores.append(avg)
        status = "✅ LULUS (>0.85)" if avg >= 0.85 else "❌ GAGAL"
        print(f"{field:20}: {avg:.4f} {status}")
        
    final_anls = np.mean(total_scores)
    print("="*40)
    print(f"RATA-RATA ANLS: {final_anls:.4f}")
    if final_anls >= 0.85:
        print("\n🎉 MODEL LULUS SYARAT NFR-007 (AKURASI >= 85%) DAN SIAP DI-DEPLOY KE PADDLEOCR-SVC / SURYA-SVC LOKAL!")

evaluate_all()
